# 00b - Base Label Playground

Exports a **small CSV** (a few hundred tweets) drawn from the **Base** partition
(`base_dataset.pkl`, ~100 000 tweets carved out by `00_hitl_data_preparation.ipynb`)
so you can start **playing around with labelling** in a spreadsheet before committing
to the full LLM-bootstrap / HITL loop.

**Why this notebook exists.** `classification_strategy.md` describes the Base partition
as a *reserve pool / source for human seed labelling*, but nothing consumes it directly.
This is the lightweight, low-stakes entry point: no model, no API, just a sample sheet.

**Input:** `Partitioned Data/{DATASET_TYPE} Data/base_dataset.pkl` — already carries
`id, text, processed_text, type, likes, retweets` (the preprocessing was done upstream
in `02_sanity_check_and_network_generation.ipynb`; the Base partition just carries it).

**Output:** `Classifiers_Data/HITL/base_label_sample.csv` — the same column form as
`03_Analysis_and_Modeling/02_extract_examples.ipynb`, plus blank `human_label` and
`notes` columns for you to fill in.

Nothing here is disjoint-partition-critical: the Base pool is a reserve, so re-running
with a different `SAMPLE_N` or sampling method is harmless.

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → base_dataset.pkl under Partitioned Data/AI Data/
# 'Art' → base_dataset.pkl under Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
datasets_folder    = BASE_PATH / 'Data Sets'
cleanedds_folder   = BASE_PATH / 'Data Sets/Cleaned Data'
partitioned_folder = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder        = datasets_folder / 'Classifiers_Data' / 'HITL'
hitl_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
%%time
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
%%time
import pandas as pd
from pathlib import Path

# Load the Base Partition

`base_dataset.pkl` is written by `00_hitl_data_preparation.ipynb`. It is the second
slice of the **attention-weighted permutation** (after the LLM Bootstrap slice), so its
natural row order is already *stratified by influence* — taking the head gives an
engagement-weighted sample for free (see the sampling cell below).

In [ ]:
%%time
BASE_PATH_PKL = partitioned_folder / 'base_dataset.pkl'
assert BASE_PATH_PKL.exists(), (
    f'Base partition not found: {BASE_PATH_PKL}. '
    f'Run 00_hitl_data_preparation.ipynb for DATASET_TYPE={DATASET_TYPE!r} first.'
)

base_df = pd.read_pickle(BASE_PATH_PKL)
print(f'Loaded base partition: {len(base_df):,} tweets')
print(f'Columns: {list(base_df.columns)}')
print(base_df['type'].value_counts().to_string())

# Sample a Few Hundred Tweets

`base_df` was already attention-weighted shuffled in `00`, so `base_df.head(SAMPLE_N)`
is an **influence-weighted** sample — it leans toward the discourse-shaping tweets
(without re-touching the very top ones, which went to the LLM Bootstrap slice).

To draw a flat random sample instead, swap the head line for
`base_df.sample(n=SAMPLE_N, random_state=42)`.

In [ ]:
%%time
SAMPLE_N = 300  # how many tweets to put in the playground sheet

sample = base_df.head(SAMPLE_N).copy()

# Labelling columns for you to fill in by hand.
sample['human_label'] = ''   # your label goes here
sample['notes']       = ''   # free-form scratch space (edge cases, questions, etc.)

# Keep the 02_extract_examples column form, then the two blank labelling columns.
CSV_COLS = ['id', 'text', 'processed_text', 'type', 'likes', 'retweets',
            'human_label', 'notes']
sample = sample[CSV_COLS]

# Collapse newlines so each tweet stays on one spreadsheet row.
for col in ('text', 'processed_text'):
    sample[col] = sample[col].astype(str).str.replace('\n', ' ', regex=False)

print(f'Sampled {len(sample):,} tweets')
att = (sample['likes'] + sample['retweets']).astype(int)
print(f'Engagement (likes+retweets): median={att.median():.0f}  mean={att.mean():.1f}  max={att.max():,}')
sample.head()

# Save the Playground Sheet

In [ ]:
%%time
OUT_CSV = hitl_folder / 'base_label_sample.csv'
sample.to_csv(OUT_CSV, index=False, encoding='utf-8')
print(f'Saved {len(sample):,} tweets → {OUT_CSV}')
print('Open it in a spreadsheet, fill in human_label / notes, and start labelling.')

# Disconnect from Runtime

In [ ]:
%%time
if not RUNNING_LOCALLY:
    from google.colab import runtime
    runtime.unassign()